# 05 — The positionwise feed-forward network

Attention mixes information across allowed positions. The MLP applies the same nonlinear feature transformation independently at each position:
$F(x)=W_{\mathrm{down}}\mathrm{GELU}(W_{\mathrm{up}}x+b_{\mathrm{up}})+b_{\mathrm{down}}$.
This equation uses column-vector notation; PyTorch Linear stores [out_features,in_features].

We will inspect expansion, nonlinearity, contraction, and the token/feature boundary.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


## 1. Expose the expanded features

Implement an MLP forward pass explicitly and compare it with the reusable module. Is the wider intermediate tensor a longer token sequence?

**Your prediction:** _Write it here before running the reference._

In [ ]:
mlp = MLP(16, 32).double()
x = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
# Your implementation: expanded = ...; activated = ...; output = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
expanded = F.linear(x, mlp.up.weight, mlp.up.bias)
activated = F.gelu(expanded)
output = F.linear(activated, mlp.down.weight, mlp.down.bias)
close(output, mlp(x))
print("Input, expansion, output:", x.shape, expanded.shape, output.shape)
print("First position, first 6 features before/after GELU:",
      expanded[0, 0, :6].detach(), activated[0, 0, :6].detach())

### Why this works

The feature axis expands from 16 to 32 while batch and time stay fixed. GELU is a smooth nonlinear transformation; it is not an attention distribution or probability normalization.

## 2. Remove the activation

Show that two affine layers without a nonlinearity collapse into one affine map, including both biases.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Your implementation: combined_weight = ...; combined_bias = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
combined_weight = mlp.down.weight @ mlp.up.weight
combined_bias = mlp.down.weight @ mlp.up.bias + mlp.down.bias
collapsed = F.linear(x, combined_weight, combined_bias)
linear_path = mlp.down(mlp.up(x))
close(collapsed, linear_path)
assert not torch.allclose(output, collapsed)
print("Affine collapse error:", float((collapsed-linear_path).abs().max().detach()))
print("Removing GELU changes output by:", float((output-collapsed).norm().detach()))

### Why this works

Expansion alone does not make the composition nonlinear. With the activation removed, the two projections are one affine function, even if their intermediate width is large.

## 3. Perturb one position and follow gradients

Change only position 3 at the MLP input, then backpropagate a loss from position 2. Which input positions can receive gradients through this isolated MLP?

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict the affected positions before running.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
changed = x.detach().clone(); changed[:, 3] += 2
original, perturbed = mlp(x), mlp(changed)
keep = torch.tensor([0, 1, 2, 4, 5])
close(original[:, keep], perturbed[:, keep])
input_gradient = torch.autograd.grad(original[:, 2].square().sum(), x, retain_graph=True)[0]
close(input_gradient[:, [0, 1, 3, 4, 5]], torch.zeros_like(input_gradient[:, [0, 1, 3, 4, 5]]))
original.square().mean().backward()
print("Input gradient norm by position:", input_gradient.norm(dim=-1))
print("Up/down projection gradient norms:",
      float(mlp.up.weight.grad.norm()), float(mlp.down.weight.grad.norm()))

### Why this works

The isolated MLP does not mix positions. Its input can nevertheless already contain context gathered by earlier attention. Shared weights accumulate gradients across positions, just like shared Q/K/V projections.

## Takeaway and evidence boundary

Next: assemble these tested pieces into one causal next-token model and inspect the complete tensor path.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.